<a href="https://colab.research.google.com/github/divyasaminathan02/Sridivyadharshini-Codeboosters-Internship-2026/blob/main/Phase_01_Data_Engineering/Phase_01_Data_Engineering/Day_03_ETL_Pandas_APIs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install requests --quiet
#The --quiet flag suppresses the installation output so our notebook stays

import pandas as pd

import numpy as np
#numpy-numerical python

import requests

import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

print("All Lbraries imported successfully")
print('pandas : {pd.__version__}')
print('requests : {requests.__version__}')

All Lbraries imported successfully
pandas : {pd.__version__}
requests : {requests.__version__}


# PART 1: ETL on Messy Sales Data

**Activity 1 - Cean messy_sales_data.csv**

Known data quality issues in this file :
1. Missing values in customer_name, quantity, category
2. Duplicate rows (orders 1001/1005 are identical)
3. Mixed Date formats : YYYY-MM-DD and DD-MM-YYYY
4. Inconsistent text case in customer_name (UPPER, lower, Title)
5. Wrong category value (keyboard labelled as Electronics)

In [5]:
raw_df = pd.read_csv('messy_sales_data.csv')
#data shape
print(f"rows in the data : {raw_df.shape[0]}")
print(f"cols in the data : {raw_df.shape[1]}")
print(f"The columns present : {raw_df.columns.tolist()}")
raw_df.columns.tolist()
raw_df.head()


rows in the data : 30
cols in the data : 9
The columns present : ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


In [6]:
print("="*50)
print("DATA QUALITY DIAGNOSIS REPORT:")
print("="*50)

#1.
print("\n[1] Missing Values per column")
print(raw_df.isnull().sum())
#isnull ->returns true or false for each cell
#sum() ->counts True (=missing) values from the cell

#2.
print(f"\n[2] Duplicate rows: {raw_df.duplicated().sum()}")

#3.
print("\n[3] Data types")
print(raw_df.dtypes)

#4.
print("\n[4] Unique Categories",raw_df['category'].unique())
print("\n[4] Sample Customers names",raw_df['customer_name'].dropna().unique()[:8])
print('[4]Sample order_date values',raw_df['order_date'].unique()[:5])





DATA QUALITY DIAGNOSIS REPORT:

[1] Missing Values per column
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] Duplicate rows: 0

[3] Data types
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] Unique Categories ['Electronics' 'Accessories' nan]

[4] Sample Customers names ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh' 'Ananya Das' 'Vikram Iyer']
[4]Sample order_date values ['2024-01-05' '2024-01-07' '2024-01-08' '2024-01-10' '07-01-2024']


In [7]:
df=raw_df.copy()
print(f"Working copy created:{df.shape} ")
print("raw_df is untouched - we can always reset by running df = raw_df.copy()")

Working copy created:(30, 9) 
raw_df is untouched - we can always reset by running df = raw_df.copy()


In [8]:
print("Before fixing nulls : ", df.isnull().sum().sum(), " total missing values")


df['customer_name'].fillna("Unknown Customer", inplace=True)

median_quantity=df['quantity'].median()

df['quantity'].fillna(median_quantity, inplace=True)

df['category'].fillna("Unknown Category", inplace=True)

print("After fixing nulls : ", df.isnull().sum().sum(), " total missing values")

Before fixing nulls :  7  total missing values
After fixing nulls :  1  total missing values


In [9]:
print(f"Before deduplicatio:{len(df)}")
print(f"Duplicated rows: {df.duplicated().sum()}")

print("\nDuplicate rows:")
print(df[df.duplicated(keep=False)][['order_id','customer_name','product','order_date']])
#keep=False marks all copies of a duplicate (not just the second+)
#this lets us see both the original and the duplicate before removing

df.drop_duplicates(inplace=True)
#identical rows in all columns will be removed

print(f"After deduplication:{len(df)} rows")
print(f'Rows removed: {len(raw_df)-len(df)}')

Before deduplicatio:30
Duplicated rows: 0

Duplicate rows:
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []
After deduplication:30 rows
Rows removed: 0


In [10]:
print('Sample dates before parsing:')
print(df['order_date'].head(8).tolist())
#Some are YYYY-MM-DD,some DD-MM-YYYY

df['order_date']=pd.to_datetime(#pd.to_datetime() converts strigs to proper datetime objects
    df['order_date'],
    dayfirst=False,   # Try YYYY-MM-DD format first (primary format)
    errors='coerce'   # If parsing fails,put NaT(Not a Time) instead of crashing,one bad date crashes the entire column
    )

nat_count=df['order_date'].isnull().sum()
print(f'\n Unparsable dates (NaT): {nat_count}')

df['year']=df['order_date'].dt.year
df['month']=df['order_date'].dt.month
df['month_name']=df['order_date'].dt.strftime('%B')
#.dt -> datetime accessor -gives access to year,month,date etc...
#strftime('%B') returns the full month name

print('\nSample dates after parsing:')
print(df[['order_date','year','month','month_name']].head(5))



Sample dates before parsing:
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13']

 Unparsable dates (NaT): 2

Sample dates after parsing:
  order_date    year  month month_name
0 2024-01-05  2024.0    1.0    January
1 2024-01-07  2024.0    1.0    January
2 2024-01-08  2024.0    1.0    January
3 2024-01-10  2024.0    1.0    January
4 2024-01-05  2024.0    1.0    January


In [11]:
print("Before standardization:",df['customer_name'].unique()[:6])
df['customer_name']=(
    df['customer_name']
    .str.strip()
    .str.title()# first letter capital in ll the words of the Title
)
#.str is pandas string accessor-applies string method to every row

print('After Standardization:',df['customer_name'].unique()[:6])

print(f"\nBefore: Keyboard rows with Electronics category:")
wrong_mask=(df['product']=='keyboard') & (df['category']=='Electronics')

print(df[wrong_mask][['product','category']])

df.loc[wrong_mask,'category']='Accessories'

print('After fix: Unique categories:',df['category'].unique())

Before standardization: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh']
After Standardization: ['Ramesh Kumar' 'Priya Nair' 'Amit Verma' 'Sunita Patel' 'Kiran Mehta'
 'Deepak Singh']

Before: Keyboard rows with Electronics category:
Empty DataFrame
Columns: [product, category]
Index: []
After fix: Unique categories: ['Electronics' 'Accessories' 'Unknown Category']


In [12]:
df['quantity']=pd.to_numeric(df['quantity'],errors='coerce').astype(int)
df['unit_price']=pd.to_numeric(df['unit_price'],errors='coerce')

df['revenue']=df['quantity'] * df['unit_price']

print("Revenue column created:")
print(df[['customer_name','product','quantity','unit_price','revenue']].head(5))
print(f"\nTotal Revenue across all orders: Rs.{df["revenue"].sum():,.0F}")

Revenue column created:
  customer_name   product  quantity  unit_price  revenue
0  Ramesh Kumar    Laptop         2       45000    90000
1    Priya Nair       NaN         1       15000    15000
2    Amit Verma  Keyboard         3        1200     3600
3  Sunita Patel   Monitor         2       22000    44000
4  Ramesh Kumar    Laptop         2       45000    90000

Total Revenue across all orders: Rs.818,000


In [13]:
print("=" * 55)
print("    POST - CLEANING VALIDATION REPORT")
print("=" * 55)

print(f"Original rows : {len(raw_df)}")
print(f"Cleaned rows : {len(df)}")
print(f"Rows removed : {len(raw_df)-len(df)} (duplicates)")
print(f"Missing values : {df.isnull().sum().sum()}")
print(f"Duplicates : {df.duplicated().sum()}")
print(f"Date nulls : {df['order_date'].isnull().sum()}")
print(f"Revenue NaN : {df['revenue'].isnull().sum()}")
print(f"Categories : {sorted(df['category'].unique())}")

print("=" * 55)

all_clean =(
    df.isnull().sum().sum() ==0 and
    df.duplicated().sum() ==0
)
print(f"Data is clean: {all_clean}")

    POST - CLEANING VALIDATION REPORT
Original rows : 30
Cleaned rows : 30
Rows removed : 0 (duplicates)
Missing values : 9
Duplicates : 0
Date nulls : 2
Revenue NaN : 0
Categories : ['Accessories', 'Electronics', 'Unknown Category']
Data is clean: False


In [14]:
product_rev=(
    df.groupby('product')['revenue']
    .sum()
    .reset_index()
    .sort_values('revenue',ascending=False)
)

print("Revenue by product:")
print(product_rev.to_string(index=False))

category_summary=df.groupby('category').agg(
    total_revenue =('revenue','sum'),
    avg_order_value =('revenue','mean'),
    num_orders=('order_id','count'),
    unique_products=('product','nunique')
).round(2).reset_index()
#agg -> applies multiple functions to multiple columns in one call

print("\n Category summary:")
print(category_summary.to_string(index=False))

Revenue by product:
   product  revenue
    Laptop   540000
   Monitor   154000
Headphones    28000
     Mouse    20800
  Keyboard    20400
    Webcam    20000
   USB Hub    19800

 Category summary:
        category  total_revenue  avg_order_value  num_orders  unique_products
     Accessories          76200          5861.54          13                4
     Electronics         697800         43612.50          16                4
Unknown Category          44000         44000.00           1                1


In [15]:
df.to_csv("clean_sales_data.csv",index=False)

print("Cleaned data saved to: clean_sales_data.csv")
print(f"Final dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print("\n ETL Pipeline for sales Data: COMPLETE")
print(" EXTRACT -> messy_sales_data.csv loaded")
print(" TRANSFORM -> nulls fixed,dupes removed,dates parsed,names standardised")
print(" LOAD -> clean_sales_data.csv saved")

Cleaned data saved to: clean_sales_data.csv
Final dataset: 30 rows x 13 columns

 ETL Pipeline for sales Data: COMPLETE
 EXTRACT -> messy_sales_data.csv loaded
 TRANSFORM -> nulls fixed,dupes removed,dates parsed,names standardised
 LOAD -> clean_sales_data.csv saved


In [16]:
API_KEY ="8b99355f7aea0530ee4a12d88e241567"

BASE_URL="https://api.openweatherapp.org/data/2.5/weather"

CITIES =['Mumbai','Delhi','Bangalore','Chennai','Hyderabad','Kolkata','Pune','Jaipur']

print(f'API configured for {len(CITIES)} cities')
print(f'Cities :{CITIES}')
print("\nIMPORTANT: Replace YOUR_API_KEY_HERE with your actual key before running")

API configured for 8 cities
Cities :['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Hyderabad', 'Kolkata', 'Pune', 'Jaipur']

IMPORTANT: Replace YOUR_API_KEY_HERE with your actual key before running


PRACTICE QUESTIONS:

1. What are the three stages of ETL? Describe each stage using an example from today's sales dataset

2. A dataframe has 500 rows. After calling df.dropna() it has 412 rows. What does this tell you?

3. Write code to remove duplicates from df where 'same row' means same customer_name AND same product

4. What is the difference between fillna(0) AND fillna(df['col'].median())? When would you prefer each?

5. Write python code to call the weather API for delhi and print the temperature in celcius

6. What is response.status_code==200 mean? What should you do when the code is 401?

In [17]:




#1. Extract:


import pandas as pd

sales_data = pd.read_csv("messy_sales_data.csv")

print(sales_data.head())


#2. Transform:

sales_data = sales_data.dropna()
sales_data = sales_data.drop_duplicates()

#Load:

sales_data.to_csv("cleaned_sales.csv", index=False)

print("ETL process completed successfully")


   order_id customer_name   product     category  quantity  unit_price  \
0      1001  Ramesh Kumar    Laptop  Electronics       2.0       45000   
1      1002    Priya Nair       NaN  Electronics       1.0       15000   
2      1003    AMIT VERMA  Keyboard  Accessories       3.0        1200   
3      1004  Sunita Patel   Monitor  Electronics       NaN       22000   
4      1005  Ramesh Kumar    Laptop  Electronics       2.0       45000   

   order_date       city    sales_rep  
0  2024-01-05     Mumbai  Anil Sharma  
1  2024-01-07      Delhi   Sunita Rao  
2  2024-01-08  Bangalore  Anil Sharma  
3  2024-01-10    Chennai   Ravi Kumar  
4  2024-01-05     Mumbai  Anil Sharma  
ETL process completed successfully


In [18]:
"""
The dataframe originally had 500 rows.

After using df.dropna(), only 412 rows remained.

This means:
500 - 412 = 88 rows contained missing (NULL/NaN) values.

dropna() removes all rows that contain at least one missing value.

Therefore, 88 rows were deleted because they had missing data.
"""

'\nThe dataframe originally had 500 rows.\n\nAfter using df.dropna(), only 412 rows remained.\n\nThis means:\n500 - 412 = 88 rows contained missing (NULL/NaN) values.\n\ndropna() removes all rows that contain at least one missing value.\n\nTherefore, 88 rows were deleted because they had missing data.\n'

In [19]:
import pandas as pd

df = df.drop_duplicates(subset=['customer_name', 'product'])

print(df)

    order_id     customer_name     product          category  quantity  \
0       1001      Ramesh Kumar      Laptop       Electronics         2   
1       1002        Priya Nair         NaN       Electronics         1   
2       1003        Amit Verma    Keyboard       Accessories         3   
3       1004      Sunita Patel     Monitor       Electronics         2   
5       1006       Kiran Mehta       Mouse       Accessories        10   
6       1007      Deepak Singh  Headphones       Electronics         2   
7       1008  Unknown Customer      Webcam       Accessories         1   
8       1009        Ananya Das      Laptop       Electronics         1   
9       1010       Vikram Iyer    Keyboard       Accessories         5   
10      1011       Pooja Gupta     Monitor       Electronics         2   
11      1012        Suresh Rao     USB Hub       Accessories         8   
12      1013       Meera Joshi      Laptop       Electronics         2   
13      1014        Arjun Nair  Headph

In [20]:
"""
fillna(0):
-----------
Replaces missing values with 0.

Example:
df['sales'] = df['sales'].fillna(0)

Use when:
- Missing value actually means zero
- Example: no sales, no profit, no discount


fillna(df['col'].median()):
---------------------------
Replaces missing values using the median of the column.

Example:
df['sales'] = df['sales'].fillna(df['sales'].median())

Use when:
- Data is numerical
- You do not want outliers to affect the replacement value
- Median is better for skewed data

Difference:
------------
fillna(0) inserts a fixed value (0)

fillna(median()) inserts a statistical value based on existing data
"""

"\nfillna(0):\n-----------\nReplaces missing values with 0.\n\nExample:\ndf['sales'] = df['sales'].fillna(0)\n\nUse when:\n- Missing value actually means zero\n- Example: no sales, no profit, no discount\n\n\nfillna(df['col'].median()):\n---------------------------\nReplaces missing values using the median of the column.\n\nExample:\ndf['sales'] = df['sales'].fillna(df['sales'].median())\n\nUse when:\n- Data is numerical\n- You do not want outliers to affect the replacement value\n- Median is better for skewed data\n\nDifference:\n------------\nfillna(0) inserts a fixed value (0)\n\nfillna(median()) inserts a statistical value based on existing data\n"

In [21]:
import requests

api_key = "6850fba354816244b7e81bbb9ef89bb2"

url = f"https://api.openweathermap.org/data/2.5/weather?q=Delhi&appid={api_key}&units=metric"

response = requests.get(url)

data = response.json()

print("Temperature in Delhi:", data['main']['temp'], "°C")

Temperature in Delhi: 38.05 °C


In [22]:
"""
response.status_code == 200 means:

The API request was successful.

The server understood the request and returned the required data successfully.


Example:
"""

import requests

response = requests.get("https://api.github.com")

print(response.status_code)

"""
If the status code is 401:

401 means "Unauthorized"

This happens when:
- API key is incorrect
- Token is missing
- User authentication failed

What should you do?
--------------------
1. Check whether the API key is correct
2. Verify login credentials
3. Make sure authentication token is valid
4. Ensure API permissions are enabled

Example:
"""

if response.status_code == 200:
    print("Request successful")

elif response.status_code == 401:
    print("Unauthorized access - Check API key or authentication")

200
Request successful


1. Weather Data ETL Project
Create a Python program that fetches weather data from the OpenWeatherMap API, cleans the data using Pandas, and stores the final output in a CSV file.

Objective :
Learn how to extract live API data, transform JSON data into a DataFrame, and load cleaned data into storage.

In [23]:
API_KEY = "6850fba354816244b7e81bbb9ef89bb2"
CITY = "Chennai"

# API URL
url = f"https://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={API_KEY}&units=metric"

# Send GET request
response = requests.get(url)

# Convert response to JSON
data = response.json()

# Display raw JSON data
print(data)

# Extract required fields
weather_data = {
    "City": [data["name"]],
    "Temperature (C)": [data["main"]["temp"]],
    "Feels Like (C)": [data["main"]["feels_like"]],
    "Humidity (%)": [data["main"]["humidity"]],
    "Pressure (hPa)": [data["main"]["pressure"]],
    "Weather": [data["weather"][0]["description"]],
    "Wind Speed (m/s)": [data["wind"]["speed"]]
}

# Create DataFrame
df = pd.DataFrame(weather_data)

# Display DataFrame
df

# Remove duplicates (if any)
df.drop_duplicates(inplace=True)

# Check for missing values
print(df.isnull().sum())

# Save to CSV
df.to_csv("weather_data.csv", index=False)

print("CSV file saved successfully!")

{'coord': {'lon': 80.2785, 'lat': 13.0878}, 'weather': [{'id': 801, 'main': 'Clouds', 'description': 'few clouds', 'icon': '02d'}], 'base': 'stations', 'main': {'temp': 34.65, 'feels_like': 41.65, 'temp_min': 33.94, 'temp_max': 35.13, 'pressure': 1004, 'humidity': 63, 'sea_level': 1004, 'grnd_level': 1003}, 'visibility': 6000, 'wind': {'speed': 6.17, 'deg': 140}, 'clouds': {'all': 20}, 'dt': 1779968538, 'sys': {'type': 2, 'id': 2104103, 'country': 'IN', 'sunrise': 1779927101, 'sunset': 1779973253}, 'timezone': 19800, 'id': 1264527, 'name': 'Chennai', 'cod': 200}
City                0
Temperature (C)     0
Feels Like (C)      0
Humidity (%)        0
Pressure (hPa)      0
Weather             0
Wind Speed (m/s)    0
dtype: int64
CSV file saved successfully!
